# Career Email Agent

This is your own project, not a course exercise — no orchestration-by-code, no orchestration-by-tools/handoffs demos. It combines two ideas:

- **From the email lab:** reading your inbox (IMAP) and sending threaded replies (SMTP).
- **From the digital-twin lab:** an LLM that answers *using your actual resume/summary as its only source of truth*, with tool-calling instead of an Agent Framework.

### How it decides what to do with each email

1. It only looks at emails that mention job/career keywords (recruiter, interview, hiring, etc.) — everything else is ignored.
2. For those, it hands the email to an LLM that has your resume + summary as context, and can call exactly one of two tools:
   - **`send_reply_tool`** — if it can answer confidently and specifically using only what's in your resume/summary, it drafts the reply itself and this sends it.
   - **`notify_me_tool`** — if the email asks something that *isn't* in your resume/summary (so it would have to guess), it does **not** reply. Instead it sends you a push notification so you can answer personally.

### Before running this
Create a folder called `twin/` next to this notebook, containing:
- `resume.pdf` — your resume (or LinkedIn PDF export)
- `summary.txt` — a short plain-text summary of you: background, current status, what you're looking for, key skills. The more specific this is, the more precisely the agent can answer on your behalf.

You'll also need these in your `.env` file (in addition to `EMAIL_ADDRESS`, `EMAIL_APP_PASSWORD`, `EMAIL_SMTP_SERVER` if you already had them from before):
```
EMAIL_IMAP_SERVER=imap.gmail.com
GOOGLE_API_KEY=your_gemini_api_key
PUSHOVER_USER=your_pushover_user_key
PUSHOVER_TOKEN=your_pushover_app_token
```


## 1. Setup

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import imaplib
import email as email_lib
from email import policy
from email.message import EmailMessage
from email.utils import parseaddr, make_msgid
import smtplib
import requests
import re
import json
import time
import os
from pathlib import Path

load_dotenv(override=True)
from datetime import datetime, timedelta


In [3]:
# --- Email credentials (same ones used for sending/reading) ---
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_IMAP_SERVER = os.getenv("EMAIL_IMAP_SERVER", "imap.gmail.com")

# --- Pushover credentials (for the "I don't know, ask Abhishek" notification) ---
PUSHOVER_USER = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

# --- Your personal context files ---
DAYS_TO_SCAN = int(os.getenv("DAYS_TO_SCAN", "7"))  # how far back to look for emails
RESUME_PATH = os.getenv("RESUME_PATH", "twin/resume.pdf")
SUMMARY_PATH = os.getenv("SUMMARY_PATH", "twin/summary.txt")

# --- Keeps this notebook from replying to / notifying about the same email twice ---
STATE_FILE = Path("handled_message_ids.json")

# Keep this True until you've tested the whole flow and are happy with it.
DRY_RUN = False

for name, val in [("EMAIL_ADDRESS", EMAIL_ADDRESS), ("EMAIL_APP_PASSWORD", EMAIL_APP_PASSWORD),
                   ("EMAIL_SMTP_SERVER", EMAIL_SMTP_SERVER), ("PUSHOVER_USER", PUSHOVER_USER),
                   ("PUSHOVER_TOKEN", PUSHOVER_TOKEN)]:
    print(f"{name}: {'set' if val else 'MISSING'}")

print(f"Resume path: {RESUME_PATH} (exists: {Path(RESUME_PATH).exists()})")
print(f"Summary path: {SUMMARY_PATH} (exists: {Path(SUMMARY_PATH).exists()})")
print(f"Dry run: {DRY_RUN}")


EMAIL_ADDRESS: set
EMAIL_APP_PASSWORD: set
EMAIL_SMTP_SERVER: set
PUSHOVER_USER: set
PUSHOVER_TOKEN: set
Resume path: C:\Users\Abhishek\Desktop\Email auto reply\twin\resume.pdf (exists: True)
Summary path: twin/summary.txt (exists: True)
Dry run: False


## 1b. Sanity checks - run these three first, before anything else

Your "no new mail" issue is almost certainly this: IMAP's `UNSEEN` search only finds
emails that are still flagged as unread. The moment you (or Gmail's own web app, or
your phone's mail app) open that test email to check if it arrived, it gets marked
**read** - so a search for `UNSEEN` comes back empty, even though the script never
actually processed it. That's the bug. The fix (further below) stops relying on the
read/unread flag entirely and instead tracks what the script itself has handled.

Run these 3 cells first, independently, to confirm each piece works on its own.

In [4]:
# SANITY CHECK 1: Can we log in and see the inbox at all?
imap = imaplib.IMAP4_SSL(EMAIL_IMAP_SERVER)
imap.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
status, data = imap.select("INBOX")
total = int(data[0])

status, unseen_data = imap.search(None, "UNSEEN")
unseen_count = len(unseen_data[0].split())

imap.logout()

print(f"Login OK. Total messages in INBOX: {total}")
print(f"Currently flagged UNSEEN: {unseen_count}  <- this is the number process_inbox() was relying on before")


Login OK. Total messages in INBOX: 24
Currently flagged UNSEEN: 21  <- this is the number process_inbox() was relying on before


In [5]:
# SANITY CHECK 2: Can we read the most recent emails, REGARDLESS of read/unread status?
imap = imaplib.IMAP4_SSL(EMAIL_IMAP_SERVER)
imap.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
imap.select("INBOX")

status, data = imap.search(None, "ALL")
all_nums = data[0].split()
last_5 = all_nums[-5:]  # most recent 5, IMAP numbers arrive in ascending order

for num in last_5:
    status, msg_data = imap.fetch(num, "(BODY.PEEK[HEADER.FIELDS (FROM SUBJECT DATE)])")
    header_text = msg_data[0][1].decode(errors="replace")
    print(header_text.strip())
    print("---")

imap.logout()
print(f"\nIf you see your test email\'s subject above, reading works fine "
      f"- the problem really was the UNSEEN filter.")


Date: Fri, 07 Aug 2026 13:39:56 -0700
Subject: You shared some Google Account data with Claude
From: Google <noreply-accounts@google.com>
---
Date: Sat, 08 Aug 2026 17:09:31 GMT
Subject: 2-Step Verification turned on
From: Google <no-reply@accounts.google.com>
---
Date: Sat, 08 Aug 2026 17:11:12 GMT
Subject: Security alert
From: Google <no-reply@accounts.google.com>
---
Date: Sat, 08 Aug 2026 10:20:02 -0700 (PDT)
From: kumarabhishek.in01@gmail.com
Subject: Sanity check - can I send?
---
From: Abhishek Kumar <kumarabhishek.logins@gmail.com>
Date: Sat, 8 Aug 2026 22:51:05 +0530
Subject: Data Engineer Opportunity - Rahul Sharma
---

If you see your test email's subject above, reading works fine - the problem really was the UNSEEN filter.


In [6]:
# SANITY CHECK 3: Can we actually send an email? (independent of everything else)
test_msg = EmailMessage()
test_msg["From"] = EMAIL_ADDRESS
test_msg["To"] = EMAIL_ADDRESS
test_msg["Subject"] = "Sanity check - can I send?"
test_msg.set_content("If you got this, SMTP sending works fine.")

with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
    server.starttls()
    server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
    server.send_message(test_msg)

print("Sent. Check your inbox for the subject: \'Sanity check - can I send?\'")


Sent. Check your inbox for the subject: 'Sanity check - can I send?'


In [7]:
# LLM client - using Gemini through an OpenAI-compatible endpoint, same as your other notebooks
MODEL_NAME = "models/gemini-3.1-flash-lite"

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


## 2. Load your personal context

This is the "ground truth" the agent is allowed to use. If something isn't in here, the agent should not answer it.

In [8]:
reader = PdfReader(RESUME_PATH)
resume_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume_text += text

print(resume_text[:500], "...")


Received a Full-Time Offer from EXL Services as an Analyst through Campus Placement (Joining: July 2026)
Completed Minor in Artificial Intelligence and Data Science at Centre for Machine Intelligence and Data Science, IITB
Professional Experience
Tribeca Developers| Business Analyst (Real Estate Analytics & CRM Systems) [Jun’25 - Jul’25]
Leading luxury real estate development firm in India, known for bringing the Trump Tower brand to Indian real estate
Analytics
&
Databases
• Organized incoming  ...


In [9]:
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = f.read()

print(summary)


CAREER & PROFESSIONAL PROFILE SUMMARY

Name
----
Abhishek Kumar

Current Status
--------------
- B.Tech graduate from IIT Bombay, completed in 2026.
- Minor in Artificial Intelligence and Data Science.
- Academic background in Mechanical Engineering, with substantial additional focus on AI/ML, data science, analytics, programming, and cloud technologies.
- Currently free and available to join immediately.
- Actively looking for full-time roles and/or immediate opportunities in Data Engineering, Data Science, Data Analytics, Business Analytics, Analytics Consulting, AI/ML, and closely related technical roles.
- Has an offer from EXL Services for an analytical role, but the joining/onboarding has been postponed to an undetermined date. Because the start date is uncertain, Abhishek is currently available and ready to join another suitable opportunity immediately.

EDUCATION
---------
Indian Institute of Technology Bombay (IIT Bombay)
- B.Tech, completed in 2026.
- Minor in Artificial Inte

In [10]:
system_prompt = f"""
# Your role

You are Abhishek Kumar's personal email-reply assistant. Abhishek receives emails
related to job opportunities, recruiters, and his career. You will be shown ONE
such email. Your job is to decide between exactly two actions, using ONLY the
information given below about Abhishek - nothing else.

1. Call `send_reply_tool` if, and only if, you can write an accurate, specific,
   and genuinely helpful reply using ONLY the information given below. Write the
   reply in first person, as if you are Abhishek himself, in a professional tone.

2. Call `notify_me_tool` if the email asks for anything - a fact, a date, a
   preference, availability, salary expectations, anything - that is not
   explicitly present in the information below. Do NOT guess, infer, or make up
   anything that isn't there. It is much better to notify Abhishek than to send
   an inaccurate or generic reply.

Always call exactly one of these two tools. Never reply in plain text without
calling a tool.

# Summary of Abhishek

{summary}

# Abhishek's Resume

{resume_text}
"""

print(system_prompt[:800], "...")



# Your role

You are Abhishek Kumar's personal email-reply assistant. Abhishek receives emails
related to job opportunities, recruiters, and his career. You will be shown ONE
such email. Your job is to decide between exactly two actions, using ONLY the
information given below about Abhishek - nothing else.

1. Call `send_reply_tool` if, and only if, you can write an accurate, specific,
   and genuinely helpful reply using ONLY the information given below. Write the
   reply in first person, as if you are Abhishek himself, in a professional tone.

2. Call `notify_me_tool` if the email asks for anything - a fact, a date, a
   preference, availability, salary expectations, anything - that is not
   explicitly present in the information below. Do NOT guess, infer, or make up
   anything that  ...


## 3. Define the two tools the agent can use

Same idea as `record_email_tool` in the digital-twin lab: plain JSON Schema describing each tool, no Agent Framework.

In [11]:
send_reply_tool_json = {
    "name": "send_reply_tool",
    "description": "Use this when you can write a confident, accurate, personalized reply to this email using only the given context about Abhishek.",
    "parameters": {
        "type": "object",
        "properties": {
            "reply_text": {
                "type": "string",
                "description": "The full reply email body, written in first person as Abhishek, professional in tone.",
            },
            "attach_resume": {
                "type": "boolean",
                "description": "True if it would help to attach Abhishek's resume to this reply (e.g. the sender is asking about his background/candidacy).",
            },
        },
        "required": ["reply_text", "attach_resume"],
        "additionalProperties": False,
    },
}

notify_me_tool_json = {
    "name": "notify_me_tool",
    "description": "Use this instead of replying when the email asks something that cannot be answered confidently using only the given context. This sends Abhishek a push notification so he can reply personally. Do not send an email reply in this case.",
    "parameters": {
        "type": "object",
        "properties": {
            "reason": {
                "type": "string",
                "description": "A short explanation of what information is missing or why you can't answer confidently.",
            },
        },
        "required": ["reason"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": send_reply_tool_json},
    {"type": "function", "function": notify_me_tool_json},
]


## 4. Tool handlers - the real Python code that runs when a tool is called

In [12]:
# Words that decide whether an incoming email even gets looked at by the agent.
# Everything else is left alone completely.
TRIGGER_KEYWORDS = [
    "job", "recruiter", "interview", "hiring", "position",
    "opportunity", "role", "vacancy", "career",
]


def contains_any_keyword(text: str, keywords: list) -> bool:
    text_lower = text.lower()
    return any(re.search(rf"\b{re.escape(kw)}\b", text_lower) for kw in keywords)


def get_email_body(msg) -> str:
    """Extract the plain-text body from a parsed email, handling multipart mail."""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain" and not part.get_filename():
                charset = part.get_content_charset() or "utf-8"
                payload = part.get_payload(decode=True)
                return payload.decode(charset, errors="replace") if payload else ""
        return ""
    else:
        charset = msg.get_content_charset() or "utf-8"
        payload = msg.get_payload(decode=True)
        return payload.decode(charset, errors="replace") if payload else ""


def load_handled_ids() -> set:
    if STATE_FILE.exists():
        return set(json.loads(STATE_FILE.read_text()))
    return set()


def save_handled_ids(ids: set) -> None:
    STATE_FILE.write_text(json.dumps(sorted(ids)))


In [13]:
def push(message: str):
    """Sends a push notification to Abhishek's phone via Pushover."""
    print(f"Push: {message}")
    if DRY_RUN:
        print("[DRY RUN] Not actually sending push.")
        return
    payload = {"user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message}
    requests.post(PUSHOVER_URL, data=payload)


In [14]:
def handle_send_reply(original_msg, reply_text: str, attach_resume: bool) -> str:
    """Sends a threaded reply to the original email. Optionally attaches the resume."""
    _, sender_addr = parseaddr(original_msg.get("Reply-To") or original_msg.get("From"))
    if not sender_addr:
        print(f"WARNING: could not determine a reply-to address for \'{original_msg.get('Subject', '')}\' "
              f"(From header: {original_msg.get('From')!r}) - skipping send.")
        return "Skipped - no valid reply-to address found"
    subject = original_msg.get("Subject", "") or ""
    if not subject.lower().startswith("re:"):
        subject = f"Re: {subject}"

    reply = EmailMessage()
    reply["From"] = EMAIL_ADDRESS
    reply["To"] = sender_addr
    reply["Subject"] = subject
    reply["In-Reply-To"] = original_msg.get("Message-ID", "")
    references = original_msg.get("References", "")
    reply["References"] = f"{references} {original_msg.get('Message-ID', '')}".strip()
    reply["Message-ID"] = make_msgid()
    reply.set_content(reply_text)

    if attach_resume:
        resume_path = Path(RESUME_PATH)
        if resume_path.exists():
            reply.add_attachment(
                resume_path.read_bytes(),
                maintype="application",
                subtype="pdf",
                filename=resume_path.name,
            )
        else:
            print(f"WARNING: resume file not found at {resume_path}, sending without attachment.")
            attach_resume = False

    if DRY_RUN:
        print(f"[DRY RUN] Would reply to {sender_addr} - subject: {subject}")
        print(f"[DRY RUN] Reply text:\n{reply_text}\n")
        return "Reply drafted (dry run - not actually sent)"

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(reply)

    print(f"Replied to {sender_addr} - subject: {subject}" + (" (resume attached)" if attach_resume else ""))
    return "Reply sent successfully"


def handle_notify(original_msg, reason: str) -> str:
    """Notifies Abhishek instead of replying, because the agent didn't have enough info."""
    sender = original_msg.get("From", "unknown sender")
    subject = original_msg.get("Subject", "") or "(no subject)"
    push(f"Career email needs your reply.\nFrom: {sender}\nSubject: {subject}\nWhy I couldn't answer: {reason}")
    return "Notification sent to Abhishek"


## 5. The agent loop

This is the manual tool-calling loop from the digital-twin lab, adapted here so the model must always end by calling one of the two tools (`tool_choice="required"`).

In [15]:
def run_agent_for_email(original_msg) -> str:
    sender = original_msg.get("From", "")
    subject = original_msg.get("Subject", "") or ""
    body = get_email_body(original_msg)

    user_content = f"From: {sender}\nSubject: {subject}\n\n{body}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=tools, tool_choice="required"
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)

        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)

            if tool_call.function.name == "send_reply_tool":
                result = handle_send_reply(original_msg, args["reply_text"], args["attach_resume"])
            elif tool_call.function.name == "notify_me_tool":
                result = handle_notify(original_msg, args["reason"])
            else:
                result = f"Unknown tool: {tool_call.function.name}"

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools, tool_choice="auto"
        )

    return response.choices[0].message.content


## 6. Reading the inbox (IMAP)

In [16]:
def fetch_recent_emails(days: int = DAYS_TO_SCAN):
    """
    Connects via IMAP and returns [(msg_number, parsed_message), ...] for every email
    from the last `days` days - regardless of whether it has been read.

    We deliberately do NOT filter by the \\Seen flag anymore: any mail client
    (Gmail web, your phone, even this script itself on a previous run) can mark an
    email as read, which would make it invisible to an UNSEEN search. Instead, we
    track what THIS SCRIPT has already handled in handled_message_ids.json (see
    process_inbox below), so read/unread status in your actual inbox no longer matters.
    """
    imap = imaplib.IMAP4_SSL(EMAIL_IMAP_SERVER)
    imap.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
    imap.select("INBOX")

    since_date = (datetime.now() - timedelta(days=days)).strftime("%d-%b-%Y")
    status, data = imap.search(None, f'(SINCE "{since_date}")')
    if status != "OK":
        imap.logout()
        return []

    messages = []
    for num in data[0].split():
        # BODY.PEEK[] fetches without changing the read/unread flag either way.
        status, msg_data = imap.fetch(num, "(BODY.PEEK[])")
        if status != "OK":
            continue
        raw_email = msg_data[0][1]
        parsed = email_lib.message_from_bytes(raw_email, policy=policy.default)
        messages.append((num, parsed))

    imap.logout()
    return messages


## 7. Putting it all together

In [17]:
def process_inbox():
    mode = "DRY RUN - nothing will actually be sent" if DRY_RUN else "LIVE - real emails/pushes will be sent"
    print(f"=== Running in {mode} mode ===\n")
    handled_ids = load_handled_ids()

    recent = fetch_recent_emails()
    if not recent:
        print(f"No emails found in the last {DAYS_TO_SCAN} day(s).")
        return

    new_count = 0

    for num, msg in recent:
        msg_id = msg.get("Message-ID", "")
        sender_addr = parseaddr(msg.get("From"))[1]

        # Already handled this exact email before (by Message-ID, not by read status)
        if msg_id and msg_id in handled_ids:
            continue

        # Loop-prevention: never act on your own emails or no-reply/mailer-daemon addresses.
        if sender_addr.lower() == (EMAIL_ADDRESS or "").lower():
            if msg_id:
                handled_ids.add(msg_id)
            continue
        if re.search(r"no-?reply|mailer-daemon|postmaster", sender_addr, re.IGNORECASE):
            if msg_id:
                handled_ids.add(msg_id)
            continue

        subject = msg.get("Subject", "") or ""
        body = get_email_body(msg)
        full_text = f"{subject}\n{body}"
        new_count += 1

        # Only career-related emails get looked at by the agent at all.
        if not contains_any_keyword(full_text, TRIGGER_KEYWORDS):
            print(f"Skipping (not career-related): \'{subject}\' from {sender_addr}")
            if msg_id:
                handled_ids.add(msg_id)
            continue

        print(f"Career email detected: \'{subject}\' from {sender_addr} - asking the agent...")
        outcome = run_agent_for_email(msg)
        print(f"Agent outcome: {outcome}\n")

        if msg_id:
            handled_ids.add(msg_id)

    save_handled_ids(handled_ids)
    print(f"Checked {len(recent)} email(s) from the last {DAYS_TO_SCAN} day(s), {new_count} were new.")


## 8. Run it - checks your inbox right now, once

In [18]:
process_inbox()


=== Running in LIVE - real emails/pushes will be sent mode ===

Career email detected: 'Data Engineer Opportunity - Rahul Sharma' from kumarabhishek.logins@gmail.com - asking the agent...
Replied to kumarabhishek.logins@gmail.com - subject: Re: Data Engineer Opportunity - Rahul Sharma (resume attached)
Agent outcome: OK. I have sent the reply to Rahul Sharma confirming your immediate availability and attaching your resume.

Checked 10 email(s) from the last 7 day(s), 1 were new.


In [ ]:
POLL_INTERVAL_SECONDS = 120  # how often to check, in seconds

while True:
    try:
        process_inbox()
    except Exception as e:
        print(f"Error during processing: {e}")
    time.sleep(POLL_INTERVAL_SECONDS)


=== Running in LIVE - real emails/pushes will be sent mode ===

Checked 10 email(s) from the last 7 day(s), 0 were new.
=== Running in LIVE - real emails/pushes will be sent mode ===

Career email detected: 'Data Engineer I Opportunity' from kumarabhishek.logins@gmail.com - asking the agent...
Replied to kumarabhishek.logins@gmail.com - subject: Re: Data Engineer I Opportunity (resume attached)
Agent outcome: None

Checked 11 email(s) from the last 7 day(s), 1 were new.
=== Running in LIVE - real emails/pushes will be sent mode ===

Checked 11 email(s) from the last 7 day(s), 0 were new.
=== Running in LIVE - real emails/pushes will be sent mode ===

Checked 11 email(s) from the last 7 day(s), 0 were new.
=== Running in LIVE - real emails/pushes will be sent mode ===

Checked 11 email(s) from the last 7 day(s), 0 were new.
=== Running in LIVE - real emails/pushes will be sent mode ===

Checked 11 email(s) from the last 7 day(s), 0 were new.
=== Running in LIVE - real emails/pushes will